#### *In This Notebook, I have done all the feature engineering, encoding, scaling, transformations and select the best model for my project with the help of metrics.*

---

# Importing Libraries.

In [27]:
import time

import numpy as np

import pandas as pd

from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, RobustScaler, PowerTransformer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# Importing Dataset.

In [2]:
data = pd.read_csv("Laptop_Prices.csv")

---

# Feature Engineering.

## 1. Removing Duplicated Data:

In [3]:
print(f"Duplicated Data: {data.duplicated().sum()}")

data.drop_duplicates(inplace=True)
print("Duplicated Data Removed Successfully!")

Duplicated Data: 2
Duplicated Data Removed Successfully!


## 2. Columns:

### a) Brand:

In [4]:
brand_counts = data['Brand'].value_counts()

data.loc[data['Brand'].map(brand_counts) <= 70, 'Brand'] = 'Other'

### b) Launch_Year:

In [5]:
data['Laptop_Age'] = 2026 - data['Launch_Year']

data.drop(columns=['Launch_Year'], inplace=True)

### c) CPU_Model:

In [6]:
data['CPU_Series'] = 'low'

mid_series_CPU = 'i5|Ultra 5|Core 5| Core 7|Ryzen 5|Ryzen 5 PRO|Ryzen AI 5|Ryzen AI 5 PRO|M5|M4|M3|X Plus'
data.loc[data['CPU_Model'].str.contains(mid_series_CPU, case=False, na=False), 'CPU_Series'] = 'mid'

high_series_CPU = 'i7|i9|Ultra 7|Ultra 9|Ryzen 7| Ryzen 7 Pro|Ryzen 9|Ryzen AI 7|Ryzen AI 9|Ryzen AI MAX|Ryzen AI MAX+|M5 Pro|M5 MAX|M4 Pro|M4 MAX|M3 Pro|M3 MAX|X Elite'
data.loc[data['CPU_Model'].str.contains(high_series_CPU, case=False, na=False), 'CPU_Series'] = 'high'


data['CPU_Segment'] = 'mid'

high_segment_CPU = 'H|HX|HS|Max|Pro|Elite'
data.loc[data['CPU_Model'].str.contains(high_segment_CPU, case=False, na=False), 'CPU_Segment'] = 'high'

data.loc[data['CPU_Model'].str.endswith('U'), 'CPU_Segment'] = 'low'
data.loc[data['CPU_Model'].str.contains('Pentium|Celeron', case=False, na=False), 'CPU_Segment'] = 'low'


data['CPU_Generation'] = 'modern'

latest_gen_CPU = 'Snapdragon|M5|AI|Core Ultra'
data.loc[data['CPU_Model'].str.contains(latest_gen_CPU, case=False, na=False), 'CPU_Generation'] = 'latest'

data.drop(columns=['CPU_Model'], inplace=True)

### d) GPU_Model:

In [7]:
data['GPU_Tier'] = 'low'

high_tier_GPU = 'RTX 4070|RTX 4070 Ti|RTX 4080|RTX 4090|RTX 5070|RTX 5070 Ti|RTX 5080|RTX 5090|M5 MAX|M4 MAX|M3 MAX|M5 Pro|M4 Pro|M3 Pro'
data.loc[data['GPU_Model'].str.contains(high_tier_GPU, case=False, na=False), 'GPU_Tier'] = 'high'

mid_tier_GPU = 'RTX 5060|RTX 5050|RTX 4060|RTX 4050|RTX 3060|RTX 3050|Intel Arc|660M|680M|760M|780M|840M|860M|880M|890M'
data.loc[data['GPU_Model'].str.contains(mid_tier_GPU, case=False, na=False), 'GPU_Tier'] = 'mid'

data.drop(columns=['GPU_Model'], inplace=True)

### e) GPU_VRAM:

In [8]:
data.loc[data['GPU_VRAM'] == 'Shared', 'GPU_VRAM'] = 0

data['GPU_VRAM'] = data['GPU_VRAM'].astype('int')

### f) Screen_Size & Resolution:

In [9]:
data[['ScreenW', 'ScreenH']] = data['Resolution'].str.split('x', expand=True).astype(int)

data['Pixel_Per_Inch'] = (np.sqrt(np.square(data['ScreenW']) + np.square(data['ScreenH'])) / data['Screen_Size']).round(2)

data.drop(columns=['Screen_Size', 'Resolution', 'ScreenW', 'ScreenH'], inplace=True)

In [10]:
data.to_csv("Laptop_Prices_Featured.csv")

## 3. Encoding:

In [11]:
# Ordinal Encoding:

ordinal_cols = ['GPU_Tier', 'CPU_Series', 'CPU_Segment', 'CPU_Generation']

ordinal_encoding = OrdinalEncoder(categories=[['low', 'mid', 'high'], ['low', 'mid', 'high'], ['low', 'mid', 'high'], ['modern', 'latest']]) 
ordinal_encoded_data = ordinal_encoding.fit_transform(data[ordinal_cols])

oe_data = pd.DataFrame(ordinal_encoded_data, columns=ordinal_cols).reset_index(drop=True)


# One-Hot Encoding:

onehot_cols = ['Brand', 'Laptop_Type', 'CPU_Brand', 'GPU_Type', 'OS']

one_hot_encoding = OneHotEncoder(drop='first', sparse_output=False)
one_hot_encoded_data = one_hot_encoding.fit_transform(data[onehot_cols])

ohe_data = pd.DataFrame(one_hot_encoded_data, columns=one_hot_encoding.get_feature_names_out()).reset_index(drop=True)

cols_left = data.drop(columns=['GPU_Tier', 'CPU_Series', 'CPU_Segment', 'CPU_Generation', 'Brand', 'Laptop_Type', 'CPU_Brand', 'GPU_Type', 'OS']).reset_index(drop=True)

final_data = pd.concat([cols_left, oe_data, ohe_data], axis=1)

In [12]:
final_data.to_csv("Laptop_Prices_Model_Ready.csv")

## 4a. Transformation:

In [13]:
# Log Transformation:

log_transform_columns = ['RAM', 'Storage']

final_data[log_transform_columns] = np.log1p(final_data[log_transform_columns])

---

# Removing Useless / Unneccessary Columns.

In [14]:
final_data.drop(columns=['Storage_Type', 'Brand_Apple'], inplace=True)

# Train Test Split.

In [15]:
X = final_data.drop(columns=['Price'])
y = np.log1p(final_data['Price'])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=69)

---

# Feature Engineering.

## 4b. Transformation:

In [16]:
# Yeo-Johnson Transformation:

yeo_johnson_cols = ['Weight', 'Pixel_Per_Inch']

yeo_johnson = PowerTransformer(method='yeo-johnson')
X_train[yeo_johnson_cols] = yeo_johnson.fit_transform(X_train[yeo_johnson_cols])
X_test[yeo_johnson_cols] = yeo_johnson.transform(X_test[yeo_johnson_cols])

## 5. Scaling:

In [17]:
# Robust Scaling:

scaler = RobustScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

---

# Model Selection.

In [18]:
def adjusted_r2(test_data, predict_data, n, k):
    r2 = r2_score(test_data, predict_data)
    return 1 - (((1 - r2) * (n - 1)) / (n - 1 - k))

def mae_percentage(y_test, y_pred):
    mae = mean_absolute_error(y_test, y_pred)
    return (mae / np.mean(y_test)) * 100 

## 1. Linear Regression:

In [19]:
Lr_model = LinearRegression()

start = time.time()
Lr_model.fit(X_train, y_train)
print(f"Time Taken For Training {(time.time() - start):.3f}s")

y_pred = Lr_model.predict(X_test)

y_pred_actual = np.expm1(y_pred)
y_test_actual = np.expm1(y_test)

print(f"\nr2 Score: {(r2_score(y_test_actual, y_pred_actual)*100):.2f}%")

print(f"\nMean Absolute Error: ₹{mean_absolute_error(y_test_actual, y_pred_actual):.2f}")
print(f"Linear Regression Model is off by {mae_percentage(y_test_actual, y_pred_actual):.2f}%")

print(f"\nRoot Mean Square Error: ₹{np.sqrt(mean_squared_error(y_test_actual, y_pred_actual)):.2f}")

print(f"\nAdjusted r2 score: {(adjusted_r2(y_test_actual, y_pred_actual, n = X_train.shape[0], k = X_train.shape[1])*100):.2f}%")

Time Taken For Training 0.306s

r2 Score: 90.61%

Mean Absolute Error: ₹22404.68
Linear Regression Model is off by 15.25%

Root Mean Square Error: ₹34670.55

Adjusted r2 score: 90.23%


## 2. Lasso Regression:

In [20]:
lasso_model = Lasso()

start = time.time()
lasso_model.fit(X_train, y_train)
print(f"Time Taken For Training {(time.time() - start):.3f}s")

y_pred = lasso_model.predict(X_test)

y_pred_actual = np.expm1(y_pred)
y_test_actual = np.expm1(y_test)

print(f"\nr2 Score: {(r2_score(y_test_actual, y_pred_actual)*100):.2f}%")

print(f"\nMean Absolute Error: ₹{mean_absolute_error(y_test_actual, y_pred_actual):.2f}")
print(f"Lasso Regression Model is off by {mae_percentage(y_test_actual, y_pred_actual):.2f}%")

print(f"\nRoot Mean Square Error: ₹{np.sqrt(mean_squared_error(y_test_actual, y_pred_actual)):.2f}")

print(f"\nAdjusted r2 score: {(adjusted_r2(y_test_actual, y_pred_actual, n = X_train.shape[0], k = X_train.shape[1])*100):.2f}%")

Time Taken For Training 0.007s

r2 Score: -7.08%

Mean Absolute Error: ₹74071.53
Lasso Regression Model is off by 50.43%

Root Mean Square Error: ₹117091.33

Adjusted r2 score: -11.41%


## 3. Ridge Regression:

In [21]:
ridge_model = Ridge()

start = time.time()
ridge_model.fit(X_train, y_train)
print(f"Time Taken For Training {(time.time() - start):.3f}s")

y_pred = ridge_model.predict(X_test)

y_pred_actual = np.expm1(y_pred)
y_test_actual = np.expm1(y_test)

print(f"\nr2 Score: {(r2_score(y_test_actual, y_pred_actual)*100):.2f}%")

print(f"\nMean Absolute Error: ₹{mean_absolute_error(y_test_actual, y_pred_actual):.2f}")
print(f"Ridge Regression Model is off by {mae_percentage(y_test_actual, y_pred_actual):.2f}%")

print(f"\nRoot Mean Square Error: ₹{np.sqrt(mean_squared_error(y_test_actual, y_pred_actual)):.2f}")

print(f"\nAdjusted r2 score: {(adjusted_r2(y_test_actual, y_pred_actual, n = X_train.shape[0], k = X_train.shape[1])*100):.2f}%")

Time Taken For Training 0.016s

r2 Score: 90.86%

Mean Absolute Error: ₹22279.92
Ridge Regression Model is off by 15.17%

Root Mean Square Error: ₹34207.08

Adjusted r2 score: 90.49%


## 4. Decision Trees:

In [22]:
dt_model = DecisionTreeRegressor()

start = time.time()
dt_model.fit(X_train, y_train)
print(f"Time Taken For Training {(time.time() - start):.3f}s")

y_pred = dt_model.predict(X_test)

y_pred_actual = np.expm1(y_pred)
y_test_actual = np.expm1(y_test)

print(f"\nr2 Score: {(r2_score(y_test_actual, y_pred_actual)*100):.2f}%")

print(f"\nMean Absolute Error: ₹{mean_absolute_error(y_test_actual, y_pred_actual):.2f}")
print(f"Decision Trees Model is off by {mae_percentage(y_test_actual, y_pred_actual):.2f}%")

print(f"\nRoot Mean Square Error: ₹{np.sqrt(mean_squared_error(y_test_actual, y_pred_actual)):.2f}")

print(f"\nAdjusted r2 score: {(adjusted_r2(y_test_actual, y_pred_actual, n = X_train.shape[0], k = X_train.shape[1])*100):.2f}%")

Time Taken For Training 0.013s

r2 Score: 88.43%

Mean Absolute Error: ₹24412.51
Decision Trees Model is off by 16.62%

Root Mean Square Error: ₹38497.11

Adjusted r2 score: 87.96%


## 5. Random Forest:

In [23]:
rf_model = RandomForestRegressor()

start = time.time()
rf_model.fit(X_train, y_train)
print(f"Time Taken For Training {(time.time() - start):.3f}s")

y_pred = rf_model.predict(X_test)

y_pred_actual = np.expm1(y_pred)
y_test_actual = np.expm1(y_test)

print(f"\nr2 Score: {(r2_score(y_test_actual, y_pred_actual)*100):.2f}%")

print(f"\nMean Absolute Error: ₹{mean_absolute_error(y_test_actual, y_pred_actual):.2f}")
print(f"Random Forest Model is off by {mae_percentage(y_test_actual, y_pred_actual):.2f}%")

print(f"\nRoot Mean Square Error: ₹{np.sqrt(mean_squared_error(y_test_actual, y_pred_actual)):.2f}")

print(f"\nAdjusted r2 score: {(adjusted_r2(y_test_actual, y_pred_actual, n = X_train.shape[0], k = X_train.shape[1])*100):.2f}%")

Time Taken For Training 0.652s

r2 Score: 91.05%

Mean Absolute Error: ₹19831.58
Random Forest Model is off by 13.50%

Root Mean Square Error: ₹33852.84

Adjusted r2 score: 90.69%


## 6. Gradient Boosting:

In [24]:
gb_model = GradientBoostingRegressor()

start = time.time()
gb_model.fit(X_train, y_train)
print(f"Time Taken For Training {(time.time() - start):.3f}s")

y_pred = gb_model.predict(X_test)

y_pred_actual = np.expm1(y_pred)
y_test_actual = np.expm1(y_test)

print(f"\nr2 Score: {(r2_score(y_test_actual, y_pred_actual)*100):.2f}%")

print(f"\nMean Absolute Error: ₹{mean_absolute_error(y_test_actual, y_pred_actual):.2f}")
print(f"Gradient Boost Model is off by {mae_percentage(y_test_actual, y_pred_actual):.2f}%")

print(f"\nRoot Mean Square Error: ₹{np.sqrt(mean_squared_error(y_test_actual, y_pred_actual)):.2f}")

print(f"\nAdjusted r2 score: {(adjusted_r2(y_test_actual, y_pred_actual, n = X_train.shape[0], k = X_train.shape[1])*100):.2f}%")

Time Taken For Training 0.229s

r2 Score: 91.72%

Mean Absolute Error: ₹19465.91
Gradient Boost Model is off by 13.25%

Root Mean Square Error: ₹32557.54

Adjusted r2 score: 91.39%


## 7. XGBoost:

In [29]:
xgb_model = XGBRegressor()

start = time.time()
xgb_model.fit(X_train, y_train)
print(f"Time Taken For Training {(time.time() - start):.3f}s")

y_pred = xgb_model.predict(X_test)

y_pred_actual = np.expm1(y_pred)
y_test_actual = np.expm1(y_test)

print(f"\nr2 Score: {(r2_score(y_test_actual, y_pred_actual)*100):.2f}%")

print(f"\nMean Absolute Error: ₹{mean_absolute_error(y_test_actual, y_pred_actual):.2f}")
print(f"Extreme Gradient Boosting Model is off by {mae_percentage(y_test_actual, y_pred_actual):.2f}%")

print(f"\nRoot Mean Square Error: ₹{np.sqrt(mean_squared_error(y_test_actual, y_pred_actual)):.2f}")

print(f"\nAdjusted r2 score: {(adjusted_r2(y_test_actual, y_pred_actual, n = X_train.shape[0], k = X_train.shape[1])*100):.2f}%")

Time Taken For Training 0.163s

r2 Score: 92.20%

Mean Absolute Error: ₹18043.95
Extreme Gradient Boosting Model is off by 12.29%

Root Mean Square Error: ₹31606.40

Adjusted r2 score: 91.88%
